# Task 3 — Schema Validation

Validate all cleaned records with the single implementation in `src/schema.py`. Every rejected row is preserved with one or more failure reasons.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from src import schema
from src.config import get_paths, load_config, year_range

config = load_config(ROOT)
paths = get_paths(config).ensure()
min_year, max_year = year_range(config)

schema.schema_table()


,column,nullable,rule
0,research_id,False,Non-empty and unique
1,university,False,Must match the source university
2,title,False,Non-empty research title
3,authors,False,Non-empty author information
4,publication_year,False,Whole year inside the project range
5,publication_date,True,Valid YYYY-MM-DD when available; same year as ...
6,abstract,True,Text when available
7,research_field,True,Text when available
8,tech_category,True,Technology category when assigned
9,journal,True,Journal or venue when available


In [2]:
kaust = pd.concat([
    pd.read_csv(paths.interim / config["sources"]["kaust_repository"]["cleaned"]),
    pd.read_csv(paths.interim / config["sources"]["kaust_crossref"]["cleaned"]),
], ignore_index=True)

kfupm = pd.read_csv(paths.interim / config["sources"]["kfupm_pure"]["cleaned"])
ksu = pd.read_csv(paths.interim / config["sources"]["ksu"]["cleaned"])

frames = {"KAUST": kaust, "KFUPM": kfupm, "KSU": ksu}


In [3]:
summary = []
for university, frame in frames.items():
    validated, rejected = schema.validate_dataset(frame, university, min_year, max_year)

    validated[schema.SCHEMA_COLUMNS].to_csv(paths.interim / f"{university}_validated.csv", index=False)
    rejected.to_csv(paths.interim / f"{university}_rejected.csv", index=False)
    schema.failures_by_rule(rejected).to_csv(
        paths.interim / f"{university}_failures_by_rule.csv", index=False
    )

    summary.append({
        "source": university,
        "cleaned": len(frame),
        "validated": len(validated),
        "rejected": len(rejected),
    })

pd.DataFrame(summary)


,source,cleaned,validated,rejected
0,KAUST,1052,938,114
1,KFUPM,441,415,26
2,KSU,41183,15622,25561


## Required vs optional fields

Required: `research_id`, `university`, `title`, `authors`, `publication_year`, `doi`, `url`, `source`.

Optional: `publication_date`, `abstract`, `research_field`, `tech_category`, `journal`.